# TopK clients

All subjects hit the same backend and the same collections, so absolute latency measures
what staging was doing that hour. Everything here is a **ratio to the baseline client**,
which is immune to that drift.

| | proto/gRPC | pgwire | ES HTTP |
|---|---|---|---|
| **rust** | `topk-rs` **(baseline)** | — | — |
| **python** | `topk-py` | `topk-sql` | `topk-es` |
| **js** | `topk-js` *(later)* | — | — |

`topk-rs` is the floor: the harness is itself Rust, so a Rust provider adds no
client-language cost — it is the protocol and nothing else. `topk-py` is **not** the
baseline, it is a client that happens to speak the same protocol and pays PyO3 plus
Python on top. Reading `topk-rs` -> `topk-py` isolates **client language**; reading
`topk-py` -> `topk-es` isolates **protocol**.

Until `topk-bench` grows a `topk-rs` provider, `topk-py` stands in as baseline and the
setup cell says so. Every "x baseline" number then still contains Python overhead and is
therefore an *under*-estimate of protocol cost.

Cross-engine comparison (TopK vs other vector DBs) lives in `bench.ipynb`.

In [ ]:
import os, glob
import polars as pl
import plotly.express as px

# The harness emits the python proto provider as "topk"; name it for what it is.
RENAME   = {"topk": "topk-py"}
ORDER    = ["topk-rs", "topk-py", "topk-sql", "topk-es", "topk-js"]
PREFERRED_BASELINE = "topk-rs"      # the real floor: Rust client, Rust harness

def repo_root():
    d = os.path.abspath(os.getcwd())
    while d != "/":
        if os.path.isfile(os.path.join(d, "local.py")):
            return d
        d = os.path.dirname(d)
    raise RuntimeError("could not locate repo root (no local.py above cwd)")

ROOT = repo_root()

# 2026-08-02 sweep, against the es-proxy build that fixes the `_source` bug. The old
# ./results-interleaved (07-30) ran while /_search ignored `_source_includes`, so every
# topk-es hit carried its 768-float embedding -- inflated, do not pool.
# Post-reset layout: one directory per sweep, manifest.json beside the parquet.
# Everything before 2026-08-02 is under stash/archive-2026-08-02/ and is NOT loaded --
# its topk-es query numbers were inflated by the `_source` bug.
QUERY_DIR  = os.path.join(ROOT, os.environ.get("BENCH_QUERY_SWEEP", "results/query-latest"))
BATCH_DIR  = os.path.join(ROOT, "results/batch-2026-08-02")

QUERY_FILES = sorted(glob.glob(os.path.join(QUERY_DIR, "*.parquet")))
# Batch sweep: results/batch-*/b<N>/*.parquet, one directory per batch size.
BATCH_FILES = sorted(glob.glob(os.path.join(BATCH_DIR, "b*", "*.parquet")))

def load(files):
    if not files:
        return None
    return (pl.concat([pl.read_parquet(f) for f in files])
              .with_columns(pl.col("provider").replace(RENAME)))

df, ing = load(QUERY_FILES), load(BATCH_FILES)

present = set()
for frame in (df, ing):
    if frame is not None:
        present |= set(frame["provider"].unique().to_list())
BASELINE = PREFERRED_BASELINE if PREFERRED_BASELINE in present else "topk-py"
if BASELINE != PREFERRED_BASELINE:
    print(f"   note: {PREFERRED_BASELINE} not in this dataset -- topk-py is the fastest client here,"
          f" but it still pays PyO3 + Python, so it is not the protocol floor.")
SIZES = [s for s in ["100k", "1m", "10m"] if df is not None and s in df["size"].unique().to_list()]

print(f"query {len(QUERY_FILES)} files, batch-sweep {len(BATCH_FILES)} files, sizes {SIZES}")
if df is not None:
    with pl.Config(tbl_rows=-1):
        print(df.group_by(["provider", "mode", "size"])
                .agg(pl.col("run_id").n_unique().alias("runs")).sort(["mode", "size", "provider"]))

In [ ]:
# One value per (group, run_id) first, so a run is the unit of repetition and the
# whiskers below are run-to-run spread.

def per_run(frame, keys, kind):
    if kind == "latency":
        return (frame.filter(pl.col("metric") == "bench.query.latency_ms")
                     .group_by(keys + ["run_id"]).agg(pl.col("value").quantile(0.99).alias("value")))
    if kind == "qps":
        return (frame.filter(pl.col("metric") == "bench.query.latency_ms")
                     .group_by(keys + ["run_id"])
                     .agg([pl.col("ts").min().alias("s"), pl.col("ts").max().alias("e"),
                           pl.len().alias("n")])
                     .with_columns((pl.col("n") /
                         ((pl.col("e") - pl.col("s")).dt.total_seconds())).alias("value"))
                     .select(keys + ["run_id", "value"]))
    if kind == "recall":
        return (frame.filter(pl.col("metric") == "bench.query.recall")
                     .group_by(keys + ["run_id"]).agg(pl.col("value").mean().alias("value")))

def agg(frame, keys, kind):
    return (per_run(frame, keys, kind).group_by(keys).agg([
        pl.col("value").mean().alias("value"),
        pl.col("value").min().alias("lo"),
        pl.col("value").max().alias("hi")]))

def vs_baseline(frame, keys, kind, pct=False):
    """Ratio to BASELINE within the same cell. pct=True -> % of baseline (QPS)."""
    a = agg(frame, keys, kind)
    other = [k for k in keys if k != "provider"]
    base = a.filter(pl.col("provider") == BASELINE).select(other + [pl.col("value").alias("b")])
    j = a.join(base, on=other, how="inner")
    m = 100 if pct else 1
    j = j.with_columns([(pl.col(c) / pl.col("b") * m).alias(c) for c in ("value", "lo", "hi")])
    # plotly error bars are offsets from the bar height, over the same runs as the mean
    return (j.with_columns([(pl.col("hi") - pl.col("value")).alias("err_plus"),
                            (pl.col("value") - pl.col("lo")).alias("err_minus"),
                            pl.col("provider").cast(pl.Enum(ORDER))])
             .sort(other + ["provider"]))

def bars(frame, x, y, title, ylab, ref=None, **kw):
    fig = px.bar(frame.to_pandas(), x=x, y=y, color="provider", barmode="group",
                 error_y="err_plus", error_y_minus="err_minus",
                 category_orders={"size": SIZES, "provider": ORDER},
                 labels={y: ylab}, title=title, **kw)
    if ref is not None:
        fig.add_hline(y=ref, line_dash="dot")
    return fig

unf = (df.filter((pl.col("mode") == "qps") & (pl.col("int_filter") == "")
                 & (pl.col("keyword_filter") == "")) if df is not None else None)

## Latency and throughput

In [ ]:
if unf is not None and len(unf):
    a = agg(unf, ["provider", "size"], "latency").with_columns([
        (pl.col("hi") - pl.col("value")).alias("err_plus"),
        (pl.col("value") - pl.col("lo")).alias("err_minus"),
        pl.col("provider").cast(pl.Enum(ORDER))])
    bars(a.sort("size"), "size", "value", "p99 latency", "p99 ms").show()

    q = agg(unf, ["provider", "size"], "qps").with_columns([
        (pl.col("hi") - pl.col("value")).alias("err_plus"),
        (pl.col("value") - pl.col("lo")).alias("err_minus"),
        pl.col("provider").cast(pl.Enum(ORDER))])
    bars(q.sort("size"), "size", "value", "Throughput", "queries / second").show()

## Throughput under concurrency

In [ ]:
if unf is not None and len(unf):
    r = (agg(unf, ["provider", "size", "concurrency"], "qps")
           .with_columns([(pl.col("hi") - pl.col("value")).alias("err_plus"),
                          (pl.col("value") - pl.col("lo")).alias("err_minus"),
                          pl.col("concurrency").cast(pl.Int32),
                          pl.col("provider").cast(pl.Enum(ORDER))])
           .sort("concurrency"))
    fig = px.line(r.to_pandas(), x="concurrency", y="value", color="provider",
                  facet_col="size", markers=True,
                  error_y="err_plus", error_y_minus="err_minus",
                  category_orders={"size": SIZES, "provider": ORDER},
                  labels={"value": "queries / second"}, title="Throughput vs concurrency")
    fig.show()

## Result-set size

The bytes-OUT axis. Response size is `k` x bytes-per-hit, so this separates a client's
per-byte cost from its per-request cost -- every other mode pins `top_k=10`, which is a
single point in payload space.

In [ ]:
ks = df.filter(pl.col("mode") == "ksweep") if df is not None else None
if ks is not None and len(ks):
    a = (agg(ks, ["provider", "size", "top_k"], "latency")
           .with_columns([(pl.col("hi") - pl.col("value")).alias("err_plus"),
                          (pl.col("value") - pl.col("lo")).alias("err_minus"),
                          pl.col("top_k").cast(pl.Int32),
                          pl.col("provider").cast(pl.Enum(ORDER))])
           .sort("top_k"))
    fig = px.line(a.to_pandas(), x="top_k", y="value", color="provider",
                  facet_col="size", markers=True, log_x=True,
                  error_y="err_plus", error_y_minus="err_minus",
                  category_orders={"size": SIZES, "provider": ORDER},
                  labels={"value": "p99 ms", "top_k": "k (log)"},
                  title="p99 latency vs result-set size")
    fig.show()
else:
    print("no ksweep data yet")

## Recall parity

Same collections, so these must coincide.

In [ ]:
rec = df.filter(pl.col("metric") == "bench.query.recall") if df is not None else None
if rec is not None and len(rec):
    a = agg(rec, ["provider", "size"], "recall").with_columns([
        (pl.col("hi") - pl.col("value")).alias("err_plus"),
        (pl.col("value") - pl.col("lo")).alias("err_minus"),
        pl.col("provider").cast(pl.Enum(ORDER))])
    fig = bars(a.sort("size"), "size", "value", "Recall by client", "recall")
    fig.update_yaxes(range=[a["lo"].min() - 0.01, a["hi"].max() + 0.01])
    fig.show()

# A dropped query never contributes a latency sample, so a mismatch means the latency
# distribution is missing exactly the slow requests.
if df is not None:
    oks  = (df.filter(pl.col("metric") == "bench.query.oks")
              .group_by(["provider", "size"]).agg(pl.col("value").sum().alias("oks")))
    lats = (df.filter(pl.col("metric") == "bench.query.latency_ms")
              .group_by(["provider", "size"]).agg(pl.len().alias("samples")))
    bad = (oks.join(lats, on=["provider", "size"])
              .filter(pl.col("oks") != pl.col("samples")))
    print("oks == latency samples: OK" if not len(bad) else f"!! MISMATCH\n{bad}")

## Write throughput vs batch size

Batch size is now the **only** knob controlling wire requests: the ES provider used to
re-split every batch on an internal `MAX_BULK_BYTES`, so the harness asking for 2000
documents quietly became ~15 HTTP requests. That constant is gone -- one logical batch
is one request -- and the shim's body limit was raised from axum's 2 MiB default to
64 MiB.

Each client should show an optimum: too small and per-request cost dominates, too large
and you pay for buffering. Where a curve simply stops, the client hit the body limit.

In [ ]:
import re
if BATCH_FILES:
    rows = []
    for f in BATCH_FILES:
        batch = int(re.search(r"/b(\d+)/", f).group(1))
        d = pl.read_parquet(f)
        lat = d.filter(pl.col("metric") == "bench.ingest.latency_ms")
        if not len(lat):
            continue
        wall = (lat["ts"].max() - lat["ts"].min()).total_seconds()
        docs = d.filter(pl.col("metric") == "bench.ingest.upserted_docs")["value"].sum()
        byts = d.filter(pl.col("metric") == "bench.ingest.upserted_bytes")["value"].sum()
        reqs = d.filter(pl.col("metric") == "bench.ingest.requests")["value"].sum()
        rows.append({"provider": d["provider"][0], "batch": batch,
                     "docs_s": docs / wall if wall else 0,
                     "MB_per_req": byts / reqs / 1e6 if reqs else 0,
                     "reqs": reqs})
    b = (pl.DataFrame(rows)
           .with_columns(pl.col("provider").replace(RENAME).cast(pl.Enum(ORDER)))
           .sort(["provider", "batch"]))

    fig = px.line(b.to_pandas(), x="batch", y="docs_s", color="provider", markers=True,
                  category_orders={"provider": ORDER}, log_x=True,
                  labels={"docs_s": "documents / second", "batch": "batch size (docs, log)"},
                  title="Ingest throughput vs batch size")
    fig.show()

    # Where each client peaks, and where it stops. A curve that ends before the others
    # ran out of body limit, not out of headroom.
    for prov in b["provider"].unique().to_list():
        s_ = b.filter(pl.col("provider") == prov).sort("docs_s", descending=True)
        top = s_.head(1)
        reached = b.filter(pl.col("provider") == prov)["batch"].max()
        print(f"  {str(prov):9} peak {top['docs_s'][0]:8,.0f} docs/s at batch={top['batch'][0]:<6} "
              f"({top['MB_per_req'][0]:.1f} MB/request), largest batch completed = {reached}")

## Effective throughput — goodput vs bytes moved

`bench.ingest.upserted_bytes` is `doc.approx_size()` summed over the **parsed**
documents, so it is protocol-independent: every client reports the same value for the
same documents. That makes it the right numerator for *useful data delivered* —
**goodput** — as distinct from bytes actually put on the wire.

The gap between them is the protocol's tax:

```
goodput      = upserted_bytes / wall          (measured, protocol-independent)
wire rate    = docs/s x wire_bytes_per_doc    (per-client constant, below)
efficiency   = goodput / wire rate = 1 / inflation
```

`wire_bytes_per_doc` is a calibration constant, not a measurement from the harness:

| client | bytes/doc | how |
|---|---|---|
| `topk-py` | ~3.4 KiB | 768 f32 = 3072 B + text + filters, binary proto — **computed**, close to `approx_size` |
| `topk-es` | 16.22 KiB | orjson ndjson of exactly the fields the provider sends — **measured over 500 real documents** |

The ES figure is the trustworthy one: it predicts the observed batch ceiling (63.4 MiB at
batch=4000 passes, 126.7 MiB at 8000 fails against the 64 MiB limit). The native figure is
computed rather than measured and should be treated as approximate — it is not derived
from anything the harness records.

In [ ]:
# Calibration constants -- see the table above. Replace with a measured
# bench.ingest.wire_bytes metric when the harness grows one.
WIRE_KIB_PER_DOC = {"topk-py": 3.4, "topk-es": 16.22}

if BATCH_FILES:
    g = []
    for f in BATCH_FILES:
        batch = int(re.search(r"/b(\d+)/", f).group(1))
        d = pl.read_parquet(f)
        lat = d.filter(pl.col("metric") == "bench.ingest.latency_ms")
        if not len(lat):
            continue
        wall = (lat["ts"].max() - lat["ts"].min()).total_seconds()
        docs = d.filter(pl.col("metric") == "bench.ingest.upserted_docs")["value"].sum()
        byts = d.filter(pl.col("metric") == "bench.ingest.upserted_bytes")["value"].sum()
        prov = RENAME.get(d["provider"][0], d["provider"][0])
        wire_kib = WIRE_KIB_PER_DOC.get(prov)
        if not wall or wire_kib is None:
            continue
        g.append({"provider": prov, "batch": batch,
                  "goodput": byts / wall / 1e6,
                  "wire": docs / wall * wire_kib * 1024 / 1e6,
                  "efficiency": (byts / docs) / (wire_kib * 1024)})
    gd = pl.DataFrame(g).with_columns(pl.col("provider").cast(pl.Enum(ORDER))).sort(["provider", "batch"])

    m = gd.unpivot(index=["provider", "batch"], on=["goodput", "wire"],
                   variable_name="kind", value_name="MB_s")
    fig = px.line(m.to_pandas(), x="batch", y="MB_s", color="provider", line_dash="kind",
                  markers=True, log_x=True, category_orders={"provider": ORDER},
                  labels={"MB_s": "MB/s", "batch": "batch size (docs, log)"},
                  title="Goodput (useful data) vs wire rate (bytes moved)")
    fig.show()

    print(f"  {'client':9} {'best goodput':>13} {'wire at that point':>19} {'efficiency':>11}")
    print("  " + "-" * 58)
    for prov in gd["provider"].unique().to_list():
        b = gd.filter(pl.col("provider") == prov).sort("goodput", descending=True).head(1)
        print(f"  {str(prov):9} {b['goodput'][0]:10.1f} MB/s {b['wire'][0]:15.1f} MB/s "
              f"{b['efficiency'][0]:10.0%}")
    print()
    print("  efficiency = useful bytes delivered per byte moved. The shortfall is what the")
    print("  protocol spends encoding a 768-float vector as decimal text.")

## Where the overhead comes from

Live measurement — needs a near-empty request to establish each client's fixed floor, which the sweep does not issue. `RUN_LIVE` guards it: running this during a sweep contaminates both.

In [ ]:
RUN_LIVE = False

if RUN_LIVE:
    import sys, time
    sys.path.insert(0, os.path.join(ROOT, "python"))
    import topk_bench as tb
    from topk_bench.providers.topk_es import TopKESProvider

    q = pl.read_parquet("/tmp/topk-bench/queries-100k.parquet")
    vec = [float(x) for x in q["dense"].head(1).to_list()[0]]
    es, nat = TopKESProvider(), tb.TopKProvider()
    coll = nat.client.collection("x-100k")

    def timed(fn, n=200):
        fn()
        t = []
        for _ in range(n):
            a = time.perf_counter_ns(); fn(); t.append((time.perf_counter_ns() - a) / 1e6)
        return sum(t) / len(t)

    rows = [
        ("topk-es", "floor (GET /)",   timed(lambda: es.client.info())),
        ("topk-es", "search k=10",     timed(lambda: es.client.search(
            index="x-100k", knn={"field": "dense_embedding", "query_vector": vec, "k": 10},
            size=10, source_includes=["text", "int_filter", "keyword_filter"]))),
        ("topk-py", "floor (count)",   timed(lambda: coll.count())),
        ("topk-py", "query k=10",      timed(lambda: nat.query("x-100k", vec, 10, None, None))),
    ]
    fl = pl.DataFrame({"provider": [r[0] for r in rows], "what": [r[1] for r in rows],
                       "ms": [r[2] for r in rows]})
    px.bar(fl.to_pandas(), x="what", y="ms", color="provider", barmode="group",
           title="Per-request floor vs full query", labels={"ms": "ms"}).show()
else:
    print("RUN_LIVE = False")